In [1]:
import sys
from pathlib import Path

PROJECT_DIR = Path.cwd().parent
sys.path.append(PROJECT_DIR.as_posix())

In [2]:
import tensorflow

from src.models import (
    create_baseline_cnn,
    create_baseline_resnet,
    create_efficientNetB0,
    create_mobileNetV2,
    create_resnet_custom,
)
from src.preprocessor import (
    encode_labels,
    get_augmentation_layer,
    normalize_images,
)
from src.training import compile_model, train_model

## Chargement des données

In [3]:
(X_train_fine, y_train_fine), (X_test_fine, y_test_fine) = (
    tensorflow.keras.datasets.cifar100.load_data(label_mode="fine")
)

# (X_train_coarse, y_train_coarse), (X_test_coarse, y_test_coarse) = (
#     tensorflow.keras.datasets.cifar100.load_data(label_mode="coarse")
# )

c:\Users\Administrateur\Documents\tp_groupe_tensorflow\.venv\Lib\site-packages\keras\src\datasets\cifar.py:18: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  d = cPickle.load(f, encoding="bytes")


## Normalisation des données

In [4]:
train_ds_fine = tensorflow.data.Dataset.from_tensor_slices((X_train_fine, y_train_fine))
test_ds_fine = tensorflow.data.Dataset.from_tensor_slices((X_test_fine, y_test_fine))

train_ds_fine_normalized = normalize_images(train_ds_fine, normalization="none")
test_ds_fine_normalized = normalize_images(test_ds_fine, normalization="none")

train_ds_fine_encoded = encode_labels(train_ds_fine_normalized, label_mode="fine")
test_ds_fine_encoded = encode_labels(test_ds_fine_normalized, label_mode="fine")

IMG_SIZE = 32


def resize_images(ds):
    def preprocess(x, y):
        return (tensorflow.image.resize(x, (IMG_SIZE, IMG_SIZE)), y)

    return ds.map(preprocess, num_parallel_calls=tensorflow.data.AUTOTUNE)


train_ds_fine_resized = resize_images(train_ds_fine_encoded)
test_ds_fine_resized = resize_images(test_ds_fine_encoded)

# train_ds_fine_mix_up = mix_up_images(train_ds_fine_resized)

BATCH_SIZE = 32

train_ds_fine_processed = (
    train_ds_fine_resized.cache()
    .shuffle(1000)
    .batch(BATCH_SIZE)
    .prefetch(tensorflow.data.AUTOTUNE)
)
test_ds_fine_processed = (
    test_ds_fine_resized.cache().batch(BATCH_SIZE).prefetch(tensorflow.data.AUTOTUNE)
)

In [5]:
# train_ds_coarse = tensorflow.data.Dataset.from_tensor_slices(
#     (X_train_coarse, y_train_coarse)
# )
# test_ds_coarse = tensorflow.data.Dataset.from_tensor_slices(
#     (X_test_coarse, y_test_coarse)
# )

# train_ds_coarse_normalized = normalize_images(train_ds_coarse, normalization="none")
# test_ds_coarse_normalized = normalize_images(test_ds_coarse, normalization="none")

# train_ds_coarse_encoded = encode_labels(train_ds_coarse_normalized, label_mode="coarse")
# test_ds_coarse_encoded = encode_labels(test_ds_coarse_normalized, label_mode="coarse")

# train_ds_coarse_mix_up = mix_up_images(train_ds_coarse_encoded)

## Création des modèles

In [6]:
augmentation_layer = get_augmentation_layer()

models = {
    "Baseline": create_baseline_cnn(
        # augmentation_layer,
    ),
    "ResNet": create_baseline_resnet(
        # augmentation_layer,
    ),
    "ResNet_custom": create_resnet_custom(
        # augmentation_layer,
    ),
    "MobileNetV2": create_mobileNetV2(
        # augmentation_layer,
    ),
    "EfficientNetB0": create_efficientNetB0(
        # augmentation_layer,
    ),
    # "hierarchical": create_hierarchical_model(augmentation_layer),
}

c:\Users\Administrateur\Documents\tp_groupe_tensorflow\.venv\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
c:\Users/Administrateur/Documents/tp_groupe_tensorflow\src\models.py:106: UserWarning: `input_shape` is undefined or non-square, or `rows` is not in [96, 128, 160, 192, 224]. Weights for input shape (224, 224) will be loaded as the default.
  base_model = keras.applications.MobileNetV2(


## Entraînement des modèles

In [10]:
img, y = next(iter(train_ds_fine))
y_cat = tensorflow.keras.utils.to_categorical(y, 100)
y_cat.shape

TensorShape([1, 100])

In [8]:
callbacks = [
    tensorflow.keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=3, restore_best_weights=True, verbose=1
    ),
    tensorflow.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", patience=3, factor=0.5, verbose=1
    ),
    # tensorflow.keras.callbacks.ModelCheckpoint(
    #     filepath="models/best_model_demo.h5",
    #     monitor="val_accuracy",
    #     save_best_only=True,
    #     mode="max",
    #     verbose=1
    # )
]

histories = {}
for name, model in models.items():
    print(f"{name} model compilation...")
    models[name] = compile_model(model=model)
    print(f"{name} model training...")
    histories[name] = train_model(
        model=model,
        train_ds=train_ds_fine_processed,
        train_val=test_ds_fine_processed,
        callbacks=callbacks,
    )
    print(f"{name} model trained")

Baseline model compilation...
Baseline model training...


ValueError: Arguments `target` and `output` must have the same rank (ndim). Received: target.shape=(None, 1, 100), output.shape=(None, 100)